# Experiment Results Explorer

This notebook auto-loads all experiment outputs under `eval/results` and helps you compare performance across runs.

## 1) Set Up Environment and Result Root Path

Import libraries and define a configurable root directory for experiment outputs.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

RESULTS_ROOT = Path("results")
COMBINED_OUTPUT_DIR = RESULTS_ROOT / "combined"
COMBINED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results root: {RESULTS_ROOT.resolve()}")

## 2) Discover Experiment Result Folders Dynamically

Recursively find experiment run folders and candidate result files without assuming a fixed number of runs.

In [ ]:
RUN_DIR_PATTERN = re.compile(
    r"^traces-(?P<ts>\d{8}-\d{6})__model-(?P<model>.+?)__temp-(?P<temp>.+?)__top-p-(?P<top_p>.+)$"
)

run_dirs = sorted(
    [p for p in RESULTS_ROOT.rglob("*") if p.is_dir() and p.name.startswith("traces-")]
)

candidate_files = [
    p for p in RESULTS_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in {".json", ".jsonl", ".csv", ".parquet"}
]

print(f"Discovered {len(run_dirs)} run directory(ies)")
print(f"Discovered {len(candidate_files)} candidate result file(s)")

if run_dirs:
    display(pd.DataFrame({"run_dir": [p.name for p in run_dirs], "path": [str(p) for p in run_dirs]}))

## 3) Load All Result Files Recursively

Read JSON/JSONL/CSV/Parquet result files, tag each row with experiment identifiers, and combine data.

In [ ]:
def _safe_read_json(path: Path) -> Any:
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _safe_read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    try:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                if isinstance(obj, dict):
                    rows.append(obj)
    except Exception:
        return []
    return rows


def _parse_run_info(run_dir_name: str) -> dict[str, Any]:
    m = RUN_DIR_PATTERN.match(run_dir_name)
    if not m:
        return {"run_ts": None, "model": None, "temperature": None, "top_p": None}
    return {
        "run_ts": m.group("ts"),
        "model": m.group("model"),
        "temperature": m.group("temp"),
        "top_p": m.group("top_p"),
    }


def _attach_meta(df: pd.DataFrame, run_dir: Path, source_file: Path) -> pd.DataFrame:
    info = _parse_run_info(run_dir.name)
    out = df.copy()
    out["run_dir"] = run_dir.name
    out["run_path"] = str(run_dir)
    out["source_file"] = str(source_file)
    for key, val in info.items():
        out[key] = val
    return out


def _normalize_prompt_versions(value: Any) -> dict[str, Any]:
    return value if isinstance(value, dict) else {}


def _prompt_signature(prompt_versions: dict[str, Any]) -> str:
    if not prompt_versions:
        return "(none)"
    parts = [f"{k}={prompt_versions[k]}" for k in sorted(prompt_versions.keys())]
    return " | ".join(parts)


run_level_frames: list[pd.DataFrame] = []
case_level_frames: list[pd.DataFrame] = []
raw_file_inventory: list[dict[str, Any]] = []

for run_dir in run_dirs:
    eval_json_path = run_dir / "foundry_eval_latest.json"
    eval_meta_path = run_dir / "eval_run_metadata.json"
    trace_meta_path = run_dir / "run_metadata.json"
    portal_json_path = run_dir / "foundry_portal_eval.json"

    eval_payload = _safe_read_json(eval_json_path) or {}
    eval_meta = _safe_read_json(eval_meta_path) or {}
    trace_meta = _safe_read_json(trace_meta_path) or {}
    portal_payload = _safe_read_json(portal_json_path) or {}

    raw_file_inventory.extend([
        {"run_dir": run_dir.name, "file": str(eval_json_path), "exists": eval_json_path.exists()},
        {"run_dir": run_dir.name, "file": str(eval_meta_path), "exists": eval_meta_path.exists()},
        {"run_dir": run_dir.name, "file": str(trace_meta_path), "exists": trace_meta_path.exists()},
        {"run_dir": run_dir.name, "file": str(portal_json_path), "exists": portal_json_path.exists()},
    ])

    summary = eval_payload.get("summary") if isinstance(eval_payload, dict) else {}
    counts = eval_payload.get("counts") if isinstance(eval_payload, dict) else {}
    portal_eval = eval_payload.get("portal_evaluation") if isinstance(eval_payload, dict) else {}

    prompt_versions = _normalize_prompt_versions(eval_meta.get("prompt_versions") if isinstance(eval_meta, dict) else None)
    if not prompt_versions:
        prompt_versions = _normalize_prompt_versions(trace_meta.get("prompt_versions") if isinstance(trace_meta, dict) else None)
    prompt_versions_json = json.dumps(prompt_versions, ensure_ascii=False, sort_keys=True) if prompt_versions else "{}"
    prompt_sig = _prompt_signature(prompt_versions)

    # Extract LLM-as-judge run-level metrics from portal_evaluation.metrics
    portal_metrics = (portal_eval or {}).get("metrics") if isinstance(portal_eval, dict) else {}
    if not isinstance(portal_metrics, dict):
        portal_metrics = {}

    run_row = {
        "timestamp_utc": (eval_meta.get("timestamp_utc") if isinstance(eval_meta, dict) else None)
        or (eval_payload.get("timestamp_utc") if isinstance(eval_payload, dict) else None)
        or (trace_meta.get("timestamp_utc") if isinstance(trace_meta, dict) else None),
        "model": (eval_meta.get("model") if isinstance(eval_meta, dict) else None)
        or (trace_meta.get("model") if isinstance(trace_meta, dict) else None),
        "temperature": (eval_meta.get("temperature") if isinstance(eval_meta, dict) else None)
        if isinstance(eval_meta, dict)
        else None,
        "top_p": (eval_meta.get("top_p") if isinstance(eval_meta, dict) else None)
        if isinstance(eval_meta, dict)
        else None,
        "prompt_versions_json": prompt_versions_json,
        "prompt_signature": prompt_sig,
        "required_tools_pass_rate": (summary or {}).get("required_tools_pass_rate"),
        "forbidden_tools_pass_rate": (summary or {}).get("forbidden_tools_pass_rate"),
        "expected_sequence_pass_rate": (summary or {}).get("expected_sequence_pass_rate"),
        "avg_tool_f1": (summary or {}).get("avg_tool_f1"),
        "row_count": (summary or {}).get("row_count"),
        "trace_rows": (counts or {}).get("trace_rows"),
        "gold_rows": (counts or {}).get("gold_rows"),
        "evaluated_rows": (counts or {}).get("evaluated_rows"),
        "unmatched_traces": (counts or {}).get("unmatched_traces"),
        "portal_status": (portal_eval or {}).get("status"),
        "portal_url": (portal_eval or {}).get("studio_url"),
        "has_portal_json": bool(portal_payload),
        # LLM-as-judge run-level averages
        "judge_intent_resolution": portal_metrics.get("intent_resolution.intent_resolution"),
        "judge_task_adherence": portal_metrics.get("task_adherence.task_adherence"),
        "judge_tool_call_accuracy": portal_metrics.get("tool_call_accuracy.gpt_tool_call_accuracy"),
        "judge_intent_resolution_pass_rate": portal_metrics.get("intent_resolution.binary_aggregate"),
        "judge_task_adherence_pass_rate": portal_metrics.get("task_adherence.binary_aggregate"),
        "judge_tool_call_accuracy_pass_rate": portal_metrics.get("tool_call_accuracy.binary_aggregate"),
    }

    run_level_frames.append(_attach_meta(pd.DataFrame([run_row]), run_dir, eval_json_path))

    rows = eval_payload.get("rows") if isinstance(eval_payload, dict) else []
    if isinstance(rows, list) and rows:
        case_rows: list[dict[str, Any]] = []
        for row in rows:
            if not isinstance(row, dict):
                continue
            det = row.get("deterministic_metrics") or {}
            ai_judge = row.get("ai_judge_metrics") or {}
            ir = ai_judge.get("intent_resolution") or {}
            ta = ai_judge.get("task_adherence") or {}
            tca = ai_judge.get("tool_call_accuracy") or {}
            case_rows.append(
                {
                    "case_id": row.get("case_id"),
                    "query": row.get("query"),
                    "prompt_signature": prompt_sig,
                    "prompt_versions_json": prompt_versions_json,
                    "required_tools_pass": det.get("required_tools_pass"),
                    "forbidden_tools_pass": det.get("forbidden_tools_pass"),
                    "expected_sequence_pass": det.get("expected_sequence_pass"),
                    "tool_f1": det.get("tool_f1"),
                    "tool_precision": det.get("tool_precision"),
                    "tool_recall": det.get("tool_recall"),
                    "required_missing": ", ".join(det.get("required_missing", [])) if isinstance(det.get("required_missing"), list) else None,
                    "forbidden_hit": ", ".join(det.get("forbidden_hit", [])) if isinstance(det.get("forbidden_hit"), list) else None,
                    "actual_tools": ", ".join(det.get("actual_tools", [])) if isinstance(det.get("actual_tools"), list) else None,
                    # LLM-as-judge case-level metrics
                    "judge_intent_resolution": ir.get("intent_resolution"),
                    "judge_intent_resolution_result": ir.get("intent_resolution_result"),
                    "judge_intent_resolution_reason": ir.get("intent_resolution_reason"),
                    "judge_task_adherence": ta.get("task_adherence"),
                    "judge_task_adherence_result": ta.get("task_adherence_result"),
                    "judge_task_adherence_reason": ta.get("task_adherence_reason"),
                    "judge_tool_call_accuracy": tca.get("tool_call_accuracy"),
                    "judge_tool_call_accuracy_result": tca.get("tool_call_accuracy_result"),
                    "judge_tool_call_accuracy_reason": tca.get("tool_call_accuracy_reason"),
                }
            )
        if case_rows:
            case_level_frames.append(_attach_meta(pd.DataFrame(case_rows), run_dir, eval_json_path))

runs_df = pd.concat(run_level_frames, ignore_index=True) if run_level_frames else pd.DataFrame()
cases_df = pd.concat(case_level_frames, ignore_index=True) if case_level_frames else pd.DataFrame()
files_df = pd.DataFrame(raw_file_inventory)

print(f"runs_df rows: {len(runs_df)}")
print(f"cases_df rows: {len(cases_df)}")
display(files_df)

## 4) Normalize and Validate Loaded Data

Standardize schema across runs so comparisons remain reliable even if output shape drifts.

In [ ]:
required_run_cols = [
    "run_dir", "timestamp_utc", "model", "temperature", "top_p", "prompt_signature", "prompt_versions_json",
    "required_tools_pass_rate", "forbidden_tools_pass_rate", "expected_sequence_pass_rate", "avg_tool_f1",
    "row_count", "evaluated_rows", "unmatched_traces",
    "judge_intent_resolution", "judge_task_adherence", "judge_tool_call_accuracy",
    "judge_intent_resolution_pass_rate", "judge_task_adherence_pass_rate", "judge_tool_call_accuracy_pass_rate",
]

for col in required_run_cols:
    if col not in runs_df.columns:
        runs_df[col] = np.nan

if not runs_df.empty:
    runs_df["timestamp_utc"] = pd.to_datetime(runs_df["timestamp_utc"], errors="coerce", utc=True)

numeric_cols = [
    "required_tools_pass_rate", "forbidden_tools_pass_rate", "expected_sequence_pass_rate", "avg_tool_f1",
    "row_count", "trace_rows", "gold_rows", "evaluated_rows", "unmatched_traces",
    "judge_intent_resolution", "judge_task_adherence", "judge_tool_call_accuracy",
    "judge_intent_resolution_pass_rate", "judge_task_adherence_pass_rate", "judge_tool_call_accuracy_pass_rate",
]

for col in numeric_cols:
    if col in runs_df.columns:
        runs_df[col] = pd.to_numeric(runs_df[col], errors="coerce")

if not cases_df.empty:
    if "timestamp_utc" not in cases_df.columns:
        run_ts_lookup = runs_df.set_index("run_dir")["timestamp_utc"].to_dict() if not runs_df.empty else {}
        cases_df["timestamp_utc"] = cases_df["run_dir"].map(run_ts_lookup)

    cases_df["timestamp_utc"] = pd.to_datetime(cases_df["timestamp_utc"], errors="coerce", utc=True)
    for col in ["tool_f1", "tool_precision", "tool_recall", "judge_intent_resolution", "judge_task_adherence", "judge_tool_call_accuracy"]:
        if col in cases_df.columns:
            cases_df[col] = pd.to_numeric(cases_df[col], errors="coerce")

runs_df = runs_df.sort_values(["timestamp_utc", "run_dir"], ascending=[False, False], na_position="last").reset_index(drop=True)

print("Run columns:", sorted(runs_df.columns.tolist()))
print("Case columns:", sorted(cases_df.columns.tolist()))
print(f"Run dataframe shape: {runs_df.shape}")
print(f"Case dataframe shape: {cases_df.shape}")

## 5) Aggregate Metrics Across Experiments

Compute summary statistics by run and by model for key performance metrics.

In [ ]:
metric_cols = [
    "required_tools_pass_rate",
    "forbidden_tools_pass_rate",
    "expected_sequence_pass_rate",
    "avg_tool_f1",
]

judge_metric_cols = [
    "judge_intent_resolution",
    "judge_task_adherence",
    "judge_tool_call_accuracy",
    "judge_intent_resolution_pass_rate",
    "judge_task_adherence_pass_rate",
    "judge_tool_call_accuracy_pass_rate",
]

all_metric_cols = metric_cols + judge_metric_cols

if runs_df.empty:
    run_summary_df = pd.DataFrame()
    model_summary_df = pd.DataFrame()
    prompt_param_summary_df = pd.DataFrame()
else:
    run_summary_df = runs_df[[
        "run_dir", "timestamp_utc", "model", "temperature", "top_p", "prompt_signature", "prompt_versions_json",
        *all_metric_cols,
        "evaluated_rows", "unmatched_traces"
    ]].copy()

    model_summary_df = (
        runs_df.groupby("model", dropna=False)[all_metric_cols]
        .agg(["mean", "median", "min", "max", "std", "count"])
        .reset_index()
    )

    prompt_param_summary_df = (
        runs_df.groupby(["model", "prompt_signature", "prompt_versions_json", "temperature", "top_p"], dropna=False)[all_metric_cols]
        .agg(["mean", "median", "min", "max", "std", "count"])
        .reset_index()
        .sort_values(("avg_tool_f1", "mean"), ascending=False, na_position="last")
    )

print("Run-level snapshot:")
display(run_summary_df)
print("Model-level aggregate stats:")
display(model_summary_df)
print("Model + prompt + params aggregate stats:")
display(prompt_param_summary_df)

## 6) Build Interactive Summary Tables

Use sortable DataFrames to inspect latest runs and pivot metrics by model.

In [ ]:
if runs_df.empty:
    print("No run data available.")
else:
    latest_runs = (
        runs_df.sort_values("timestamp_utc", ascending=False)
        .drop_duplicates(subset=["run_dir"])
        [["run_dir", "timestamp_utc", "model", "temperature", "top_p", "prompt_signature", "prompt_versions_json",
          *metric_cols, *judge_metric_cols, "evaluated_rows", "portal_status"]]
    )

    model_pivot = runs_df.pivot_table(
        index="model",
        values=all_metric_cols,
        aggfunc="mean",
    ).sort_values("avg_tool_f1", ascending=False)

    prompt_param_pivot = runs_df.pivot_table(
        index=["model", "prompt_signature", "prompt_versions_json", "temperature", "top_p"],
        values=all_metric_cols,
        aggfunc="mean",
    ).sort_values("avg_tool_f1", ascending=False)

    print("Latest run snapshot:")
    display(latest_runs)
    print("Average metric pivot by model:")
    display(model_pivot)
    print("Average metric pivot by model + prompt + params:")
    display(prompt_param_pivot)

## 7) Visualize Experiment Performance

Plot deterministic metric trends and LLM-as-judge metric comparisons across models.

In [ ]:
if runs_df.empty:
    print("No data to visualize.")
else:
    chart_df = runs_df.sort_values("timestamp_utc").copy()

    # --- Deterministic metrics over time ---
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    fig.suptitle("Deterministic Metrics Over Time", fontsize=14, y=1.0)
    for ax, metric in zip(axes.flatten(), metric_cols):
        for model_name, grp in chart_df.groupby("model", dropna=False):
            ax.plot(grp["timestamp_utc"], grp[metric], marker="o", label=str(model_name))
        ax.set_title(metric)
        ax.set_ylim(0, 1.05)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, ncol=max(1, len(labels)))
    fig.autofmt_xdate(rotation=30)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # --- LLM-as-judge score metrics over time ---
    judge_score_cols = ["judge_intent_resolution", "judge_task_adherence", "judge_tool_call_accuracy"]
    available_judge_scores = [c for c in judge_score_cols if c in chart_df.columns and chart_df[c].notna().any()]
    if available_judge_scores:
        fig2, axes2 = plt.subplots(1, len(available_judge_scores), figsize=(6 * len(available_judge_scores), 5), squeeze=False)
        fig2.suptitle("LLM-as-Judge Scores Over Time", fontsize=14, y=1.02)
        for ax, metric in zip(axes2.flatten(), available_judge_scores):
            for model_name, grp in chart_df.groupby("model", dropna=False):
                ax.plot(grp["timestamp_utc"], grp[metric], marker="s", label=str(model_name))
            ax.set_title(metric.replace("judge_", "").replace("_", " ").title())
            if "task_adherence" in metric:
                ax.set_ylim(0, 1.05)
            else:
                ax.set_ylim(0, 5.25)
            ax.tick_params(axis="x", rotation=30)
        handles2, labels2 = axes2[0, 0].get_legend_handles_labels()
        if handles2:
            fig2.legend(handles2, labels2, ncol=max(1, len(labels2)))
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

    # --- LLM-as-judge pass rates over time ---
    judge_pass_cols = ["judge_intent_resolution_pass_rate", "judge_task_adherence_pass_rate", "judge_tool_call_accuracy_pass_rate"]
    available_judge_pass = [c for c in judge_pass_cols if c in chart_df.columns and chart_df[c].notna().any()]
    if available_judge_pass:
        fig3, axes3 = plt.subplots(1, len(available_judge_pass), figsize=(6 * len(available_judge_pass), 5), squeeze=False)
        fig3.suptitle("LLM-as-Judge Pass Rates Over Time", fontsize=14, y=1.02)
        for ax, metric in zip(axes3.flatten(), available_judge_pass):
            for model_name, grp in chart_df.groupby("model", dropna=False):
                ax.plot(grp["timestamp_utc"], grp[metric], marker="^", label=str(model_name))
            ax.set_title(metric.replace("judge_", "").replace("_pass_rate", "").replace("_", " ").title() + " Pass Rate")
            ax.set_ylim(0, 1.05)
            ax.tick_params(axis="x", rotation=30)
        handles3, labels3 = axes3[0, 0].get_legend_handles_labels()
        if handles3:
            fig3.legend(handles3, labels3, ncol=max(1, len(labels3)))
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

    # --- Box plot: avg_tool_f1 by model ---
    plt.figure(figsize=(10, 5))
    box_data = [
        runs_df.loc[runs_df["model"] == m, "avg_tool_f1"].dropna().values
        for m in runs_df["model"].dropna().unique()
    ]
    box_labels = [str(m) for m in runs_df["model"].dropna().unique()]
    if box_data and box_labels:
        plt.boxplot(box_data, tick_labels=box_labels)
        plt.title("avg_tool_f1 distribution by model")
        plt.xticks(rotation=20, ha="right")
        plt.tight_layout()
        plt.show()

    # --- Bar chart: Mean deterministic metrics by model ---
    model_avg = runs_df.groupby("model", dropna=False)[metric_cols].mean()
    model_avg.plot(kind="bar", figsize=(10, 5))
    plt.title("Mean Deterministic Metrics by Model")
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

    # --- Bar chart: Mean LLM-as-judge scores by model ---
    judge_score_available = [c for c in judge_score_cols if c in runs_df.columns and runs_df[c].notna().any()]
    if judge_score_available:
        model_judge_avg = runs_df.groupby("model", dropna=False)[judge_score_available].mean()
        model_judge_avg.columns = [c.replace("judge_", "").replace("_", " ").title() for c in model_judge_avg.columns]
        model_judge_avg.plot(kind="bar", figsize=(10, 5))
        plt.title("Mean LLM-as-Judge Scores by Model")
        plt.ylim(0, 5.25)
        plt.tight_layout()
        plt.show()

    # --- Bar chart: Mean LLM-as-judge pass rates by model ---
    judge_pass_available = [c for c in judge_pass_cols if c in runs_df.columns and runs_df[c].notna().any()]
    if judge_pass_available:
        model_judge_pass = runs_df.groupby("model", dropna=False)[judge_pass_available].mean()
        model_judge_pass.columns = [c.replace("judge_", "").replace("_pass_rate", "").replace("_", " ").title() + " Pass Rate" for c in model_judge_pass.columns]
        model_judge_pass.plot(kind="bar", figsize=(10, 5))
        plt.title("Mean LLM-as-Judge Pass Rates by Model")
        plt.ylim(0, 1.05)
        plt.tight_layout()
        plt.show()

## 8) Highlight Best and Worst Runs

Rank experiments by objective metrics and flag potential outliers/failures.

In [ ]:
TOP_N = 5
OBJECTIVE_COL = "avg_tool_f1"  # maximize

if runs_df.empty or OBJECTIVE_COL not in runs_df.columns:
    print("No ranking data available.")
else:
    ranking_df = runs_df.dropna(subset=[OBJECTIVE_COL]).copy()
    best_runs = ranking_df.nlargest(TOP_N, OBJECTIVE_COL)[
        ["run_dir", "timestamp_utc", "model", "prompt_signature", "prompt_versions_json", "temperature", "top_p", OBJECTIVE_COL, "required_tools_pass_rate", "expected_sequence_pass_rate"]
    ]
    worst_runs = ranking_df.nsmallest(TOP_N, OBJECTIVE_COL)[
        ["run_dir", "timestamp_utc", "model", "prompt_signature", "prompt_versions_json", "temperature", "top_p", OBJECTIVE_COL, "required_tools_pass_rate", "expected_sequence_pass_rate"]
    ]

    # Build failure flag columns including LLM-as-judge info
    failure_flag_cols = [
        "run_dir", "timestamp_utc", "model", "prompt_signature", "prompt_versions_json", "temperature", "top_p",
        "required_tools_pass_rate", "expected_sequence_pass_rate", "unmatched_traces",
    ]
    judge_failure_cols = [c for c in [
        "judge_intent_resolution", "judge_task_adherence", "judge_tool_call_accuracy",
        "judge_intent_resolution_pass_rate", "judge_task_adherence_pass_rate", "judge_tool_call_accuracy_pass_rate",
    ] if c in runs_df.columns]
    failure_flag_cols.extend(judge_failure_cols)

    failure_flags = runs_df[
        (runs_df["required_tools_pass_rate"] < 1.0)
        | (runs_df["expected_sequence_pass_rate"] < 1.0)
        | (runs_df["unmatched_traces"] > 0)
    ][failure_flag_cols]

    print("Top runs:")
    display(best_runs)
    print("Bottom runs:")
    display(worst_runs)
    print("Failure/outlier flags (with LLM-as-judge context):")
    display(failure_flags)

if not cases_df.empty:
    # Build case failure columns including LLM-as-judge info
    case_failure_cols = ["run_dir", "model", "prompt_signature", "prompt_versions_json", "case_id", "tool_f1",
                         "required_missing", "forbidden_hit", "actual_tools", "query"]
    judge_case_cols = [c for c in [
        "judge_intent_resolution", "judge_intent_resolution_result", "judge_intent_resolution_reason",
        "judge_task_adherence", "judge_task_adherence_result", "judge_task_adherence_reason",
        "judge_tool_call_accuracy", "judge_tool_call_accuracy_result", "judge_tool_call_accuracy_reason",
    ] if c in cases_df.columns]
    case_failure_cols.extend(judge_case_cols)

    case_failures = cases_df[
        (cases_df["required_tools_pass"] == False)
        | (cases_df["expected_sequence_pass"] == False)
    ][case_failure_cols]
    print("Case-level failures (with LLM-as-judge context):")
    display(case_failures.sort_values(["run_dir", "case_id"], ascending=[False, True]))

## 9) Save Combined Outputs for Reuse

Persist cleaned combined data and summary tables so future analysis is fast and reproducible.

In [ ]:
combined_runs_csv = COMBINED_OUTPUT_DIR / "runs_combined.csv"
combined_cases_csv = COMBINED_OUTPUT_DIR / "cases_combined.csv"
run_summary_csv = COMBINED_OUTPUT_DIR / "run_summary.csv"
model_summary_csv = COMBINED_OUTPUT_DIR / "model_summary.csv"

if not runs_df.empty:
    runs_df.to_csv(combined_runs_csv, index=False)
    if 'run_summary_df' in globals() and isinstance(run_summary_df, pd.DataFrame):
        run_summary_df.to_csv(run_summary_csv, index=False)

if not cases_df.empty:
    cases_df.to_csv(combined_cases_csv, index=False)

if 'model_summary_df' in globals() and isinstance(model_summary_df, pd.DataFrame):
    model_summary_df.to_csv(model_summary_csv, index=False)

print("Saved outputs:")
for p in [combined_runs_csv, combined_cases_csv, run_summary_csv, model_summary_csv]:
    print(f" - {p} (exists={p.exists()})")